In [ ]:
!pip install transformers
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `magang5` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-crede

In [ ]:
import torch
import torch
import pandas as pd
from tqdm import tqdm

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("willywonka19/indobert-sentiment-rs-5")
tokenizer = BertTokenizer.from_pretrained("ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# set model to eval
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# import data
df = pd.read_csv("/content/dataset_final.csv")
df.head()

,title,stars,text,pelayanan,fasilitas,predicted_labels
0,RS Brayat Minulya,5,telah dirawat di sini selama beberapa hari di ...,1,1,"['pelayanan', 'fasilitas']"
1,RS Brayat Minulya,5,bersih dan rapi petugas peeawatdokter ditanya ...,1,1,"['pelayanan', 'fasilitas']"
2,RS Brayat Minulya,5,terimakasih atas pelayanan dan perawatan ibu s...,1,0,['pelayanan']
3,RS Brayat Minulya,5,pelayanan diruang yosef bagus perawat ramah ra...,1,0,['pelayanan']
4,RS Brayat Minulya,5,tempatnya bagusrapi nyaman sekali ruangan nya ...,1,1,"['pelayanan', 'fasilitas']"


In [ ]:
# buat mapping
idx2label = {0: "negative", 1: "positive"}

In [ ]:
def predict_sentiment(df, model, tokenizer, text_column="text", batch_size=16, max_len=512):
    predictions = []

    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_texts = df[text_column].iloc[i:i+batch_size].tolist()
            encoding = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            outputs = model(**encoding)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).tolist()
            predictions.extend(preds)

    # Tambahkan kolom 'labels' dengan hasil prediksi ('pos' atau 'neg')
    df['labels'] = [idx2label[p] for p in predictions]
    return df


In [ ]:
# Lakukan prediksi
df_pred = predict_sentiment(df, model, tokenizer)
print(df_pred)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


                                 title  stars  \
0                    RS Brayat Minulya      5   
1                    RS Brayat Minulya      5   
2                    RS Brayat Minulya      5   
3                    RS Brayat Minulya      5   
4                    RS Brayat Minulya      5   
...                                ...    ...   
7634  Rumah Sakit Umum Pusat Surakarta      4   
7635  Rumah Sakit Umum Pusat Surakarta      4   
7636  Rumah Sakit Umum Pusat Surakarta      4   
7637  Rumah Sakit Umum Pusat Surakarta      5   
7638  Rumah Sakit Umum Pusat Surakarta      5   

                                                   text  pelayanan  fasilitas  \
0     telah dirawat di sini selama beberapa hari di ...          1          1   
1     bersih dan rapi petugas peeawatdokter ditanya ...          1          1   
2     terimakasih atas pelayanan dan perawatan ibu s...          1          0   
3     pelayanan diruang yosef bagus perawat ramah ra...          1          0   
4     

In [ ]:
df_pred['labels'].value_counts()

,count
labels,
positive,7621
negative,18


In [ ]:
df_pred

,title,stars,text,pelayanan,fasilitas,predicted_labels,labels
0,RS Brayat Minulya,5,telah dirawat di sini selama beberapa hari di ...,1,1,"['pelayanan', 'fasilitas']",positive
1,RS Brayat Minulya,5,bersih dan rapi petugas peeawatdokter ditanya ...,1,1,"['pelayanan', 'fasilitas']",positive
2,RS Brayat Minulya,5,terimakasih atas pelayanan dan perawatan ibu s...,1,0,['pelayanan'],positive
3,RS Brayat Minulya,5,pelayanan diruang yosef bagus perawat ramah ra...,1,0,['pelayanan'],positive
4,RS Brayat Minulya,5,tempatnya bagusrapi nyaman sekali ruangan nya ...,1,1,"['pelayanan', 'fasilitas']",positive
...,...,...,...,...,...,...,...
7634,Rumah Sakit Umum Pusat Surakarta,4,lama banget pelayanan untuk pasien baru ibu ny...,1,0,['pelayanan'],positive
7635,Rumah Sakit Umum Pusat Surakarta,4,hospital yg sangat rapi pelayanan bagus,1,1,"['pelayanan', 'fasilitas']",positive
7636,Rumah Sakit Umum Pusat Surakarta,4,khusus paru,0,0,[],positive
7637,Rumah Sakit Umum Pusat Surakarta,5,rs paru,0,0,[],positive


In [ ]:
# simpan
df_pred.to_csv('result.csv', index = False)